# Revision experiments B+C - one run
**IJIES 20264768.** Everything still open after notebook A, in a single session (**~6-7 h**).

| Phase | Answers | What it does |
|---|---|---|
| **1** | R2.6 | **Copy gate FIXED** (notebook A trained it with loss=0). gate vs no-gate x 3 seeds + paired bootstrap |
| **2** | R2.7 | stage-2 vs **joint/rehearsal** evaluated on **learner text (BEA dev)**, not just in-domain |
| **3** | **R2.3** | **Official CoNLL-2014 M² scorer** (the "potentially fatal" comment) alongside ERRANT |
| **4** | R1.4 / R1.10 / R2.1 | **2024 LLM baseline** (Qwen2.5-Instruct, zero- and few-shot) head-to-head on *identical* splits + *identical* scorers |
| **5** | R2.7 | BEA-2019 dev **per CEFR level** (A / B / C / N) |

### Run
1. Add Input -> Datasets -> **`dariocioni/c4200m`**
2. *(optional but recommended)* upload your **`t5v3_results_exportfinalResluts.zip`** contents as a Kaggle dataset
   so the **full-scale** paper predictions get the official M² score too. Without it, phase 3 still
   scores the matched-budget model and the LLM baseline.
3. GPU **T4 x2**, Internet **ON**, `Run All`
4. Send back **`revBC_results.zip`**

`QUICK_TEST = True` -> ~30 min end-to-end check.


In [ ]:
# ============================== CONFIG ==============================
import os, json, time, math, random, gc, sys, subprocess, zipfile, tarfile, urllib.request, re, shutil
from pathlib import Path
NB = "revBC_2026_08_04"; OUT = Path("/kaggle/working"); SEED = 42
QUICK_TEST = False

RUN_P1_GATE, RUN_P2_TRANSFER, RUN_P3_M2, RUN_P4_LLM, RUN_P5_CEFR = True, True, True, True, True

TRAIN_N = 150_000 if not QUICK_TEST else 3_000     # matched to notebook A
VAL_N, EVAL_N = 2_000, 3_000
if QUICK_TEST: VAL_N, EVAL_N = 300, 300
PER_DEVICE_BATCH, GRAD_ACCUM, EPOCHS = 16, 4, 1
LEARNING_RATE = 3e-4
LABEL_SMOOTHING = 0.0        # THE FIX: keeps `labels` in inputs so the gate computes its own loss
SRC_DROPOUT_P = 0.05
MAX_INPUT_LEN = MAX_TARGET_LEN = 128
PREFIX = "Correct the grammar in this sentence: "
GEN_BATCH = 64
BASE_GEN = dict(max_length=128, num_beams=5, no_repeat_ngram_size=3, length_penalty=1.0)
SEEDS = [42, 1337, 2024]
STAGE2_EPOCHS, STAGE2_LR = 3, 1e-4
PER_SHARD = 20_000 if not QUICK_TEST else 1_500
MIN_WORDS, MAX_WORDS, MAX_LEN_DIFF = 4, 50, 10
MIN_BAG_OVERLAP, MAX_NONASCII_RATIO = 0.5, 0.25

LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"    # Sept-2024 model, 19x larger than our 77M
LLM_MAX_NEW, LLM_BATCH = 96, 32
BEA_URL   = "https://www.cl.cam.ac.uk/research/nl/bea2019st/data/wi+locness_v2.1.bea19.tar.gz"
CONLL_URL = "https://www.comp.nus.edu.sg/~nlp/conll14st/conll14st-test-data.tar.gz"
M2_URL    = "https://www.comp.nus.edu.sg/~nlp/sw/m2scorer.tar.gz"

GLOBAL_BUDGET_H = 10.5; T0 = time.time()
def hours_left(): return GLOBAL_BUDGET_H - (time.time()-T0)/3600
def budget_ok(n=0.5):
    ok = hours_left() > n
    if not ok: print(f"!! budget guard: {hours_left():.2f}h left, need {n}h -> skip")
    return ok
RES_PATH = OUT/"revBC_results.json"
RESULTS = json.loads(RES_PATH.read_text()) if RES_PATH.exists() else {}
def save(): RES_PATH.write_text(json.dumps(RESULTS, indent=1, default=str))
print(NB, "| label_smoothing =", LABEL_SMOOTHING, "| QUICK_TEST =", QUICK_TEST)


In [ ]:
# ===== FULL-SCALE paper predictions, embedded (no Kaggle dataset needed) =====
# These are the exact outputs of the two-stage FLAN-T5-small system reported in the
# submitted paper (ERRANT: BEA-2019 dev F0.5 = 0.429, CoNLL-2014 F0.5 = 0.453).
# Phase 3 rescores the CoNLL outputs with the OFFICIAL M2 scorer (reviewer R2.3);
# Phase 5 breaks the BEA outputs down per CEFR level (reviewer R2.7).
import gzip, base64
_CONLL_PRED_B64 = "..."
_BEA_PRED_B64 = "..."
def _unblob(b): return gzip.decompress(base64.b64decode(b)).decode().split("\n")
FULL_CONLL_PRED = _unblob(_CONLL_PRED_B64)
FULL_BEA_PRED   = _unblob(_BEA_PRED_B64)
assert len(FULL_CONLL_PRED) == 1312, len(FULL_CONLL_PRED)
assert len(FULL_BEA_PRED) == 4384, len(FULL_BEA_PRED)
print(f"embedded full-scale predictions: CoNLL {len(FULL_CONLL_PRED)}, BEA dev {len(FULL_BEA_PRED)}")


In [ ]:
!pip install -q transformers accelerate sentencepiece errant pyarrow 2>/dev/null
import torch, transformers, numpy as np, pandas as pd
from collections import Counter
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq, set_seed)
from torch.utils.data import Dataset
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s); set_seed(s)
seed_all(SEED); DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("transformers", transformers.__version__, "| GPUs", torch.cuda.device_count())


## Patched copy-gate module

In [ ]:
%%writefile /kaggle/working/t5_copy_gate_fixed.py
# T5 with a pointer-generator copy gate (port of Model 3's mechanism).
# This class lives in a real module file (not the notebook) because recent
# transformers versions inspect the defining module's source file during
# from_pretrained; classes defined in a notebook (__main__) have no __file__.
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import T5ForConditionalGeneration
from transformers.modeling_outputs import Seq2SeqLMOutput


class T5WithCopyGate(T5ForConditionalGeneration):
    """FLAN-T5 + pointer-generator copy gate.

    P(w) = p_gen * softmax(lm_logits)[w] + (1 - p_gen) * sum_{i: src_i == w} attn_i
    attn  = head-averaged cross-attention of the last decoder layer
    p_gen = sigmoid(W_g [decoder_state ; attention_context] + b_g)

    The returned `logits` are log-probabilities of the mixture. The mixture sums
    to ~1, so log_softmax over it is (numerically) the identity: HF generate(),
    beam search, and the Trainer's label smoothing work unchanged.
    """

    def __init__(self, config):
        super().__init__(config)
        self.copy_gate = nn.Linear(2 * config.d_model, 1)
        nn.init.zeros_(self.copy_gate.weight)
        nn.init.constant_(self.copy_gate.bias, 2.0)
        self._last_pgen_mean = None

    def forward(self, input_ids=None, attention_mask=None, decoder_input_ids=None,
                labels=None, copy_src_ids=None, **kwargs):
        kwargs.pop("num_items_in_batch", None)   # transformers>=4.46 injects this
        if labels is not None and decoder_input_ids is None and kwargs.get("decoder_inputs_embeds") is None:
            decoder_input_ids = self._shift_right(labels)
        kwargs["output_attentions"] = True
        kwargs["output_hidden_states"] = True
        kwargs["return_dict"] = True
        outputs = super().forward(input_ids=input_ids, attention_mask=attention_mask,
                                  decoder_input_ids=decoder_input_ids, labels=None, **kwargs)

        if copy_src_ids is None:
            copy_src_ids = input_ids

        if copy_src_ids is None:
            # No source ids available (plain LM usage): keep the vanilla distribution.
            new_logits = outputs.logits
        else:
            attn = outputs.cross_attentions[-1].mean(dim=1).float()        # (B, T, S)
            if attention_mask is not None:
                attn = attn * attention_mask.unsqueeze(1).float()          # zero padded src
            enc_h = outputs.encoder_last_hidden_state.float()              # (B, S, D)
            dec_h = outputs.decoder_hidden_states[-1].float()              # (B, T, D)
            context = torch.bmm(attn, enc_h)                               # (B, T, D)
            p_gen = torch.sigmoid(self.copy_gate(torch.cat([dec_h, context], dim=-1)).float())
            self._last_pgen_mean = float(p_gen.mean().detach())

            probs = torch.softmax(outputs.logits.float(), dim=-1)          # (B, T, V)
            probs = probs * p_gen
            copy_w = attn * (1.0 - p_gen)                                  # (B, T, S)
            idx = copy_src_ids.unsqueeze(1).expand(-1, probs.size(1), -1)  # (B, T, S)
            probs.scatter_add_(2, idx, copy_w)
            new_logits = torch.log(probs + 1e-9)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(new_logits.view(-1, new_logits.size(-1)),
                                   labels.view(-1).to(new_logits.device), ignore_index=-100)

        return Seq2SeqLMOutput(
            loss=loss,
            logits=new_logits,
            past_key_values=outputs.past_key_values,
            decoder_hidden_states=outputs.decoder_hidden_states,
            decoder_attentions=outputs.decoder_attentions,
            cross_attentions=outputs.cross_attentions,
            encoder_last_hidden_state=outputs.encoder_last_hidden_state,
            encoder_hidden_states=outputs.encoder_hidden_states,
            encoder_attentions=outputs.encoder_attentions,
        )

    def prepare_inputs_for_generation(self, input_ids, **kwargs):
        model_inputs = super().prepare_inputs_for_generation(input_ids, **kwargs)
        if kwargs.get("copy_src_ids") is not None:
            model_inputs["copy_src_ids"] = kwargs["copy_src_ids"]
        return model_inputs

In [ ]:
sys.path.insert(0, "/kaggle/working")
import importlib, t5_copy_gate_fixed; importlib.reload(t5_copy_gate_fixed)
from t5_copy_gate_fixed import T5WithCopyGate
print("patched T5WithCopyGate imported")


## Shared helpers: data, metrics, ERRANT

In [ ]:
def norm_ws(s): return re.sub(r"\s+"," ",str(s)).strip()
MOJIBAKE=("Ã","â€","Â","ï¿½")
RE_MARKUP=re.compile(r"<[^>]+>|&[a-z]{2,6};|https?://|\{\{|\}\}")
RE_PUNCTRUN=re.compile(r"[!?.,;:]{3,}"); RE_REPEAT=re.compile(r"\b(\w+)\s+\1\b",re.I)
def nonascii_ratio(s): return sum(ord(c)>127 for c in s)/max(len(s),1)
def clean_reason(src,tgt):
    ls,lt=len(src.split()),len(tgt.split())
    if not (MIN_WORDS<=ls<=MAX_WORDS and MIN_WORDS<=lt<=MAX_WORDS): return "length"
    if src==tgt: return "identical"
    if abs(ls-lt)>MAX_LEN_DIFF: return "len_diff"
    a,b=Counter(src.lower().split()),Counter(tgt.lower().split())
    if sum((a&b).values())/max(min(ls,lt),1)<MIN_BAG_OVERLAP: return "low_overlap"
    if RE_PUNCTRUN.search(tgt): return "punct_run_tgt"
    if RE_MARKUP.search(src) or RE_MARKUP.search(tgt): return "markup"
    if RE_REPEAT.search(tgt): return "repeat_word_tgt"
    if any(m in src for m in MOJIBAKE) or any(m in tgt for m in MOJIBAKE): return "mojibake"
    if nonascii_ratio(tgt)>MAX_NONASCII_RATIO: return "nonascii_tgt"
    return None

C4=Path("/kaggle/input/datasets/dariocioni/c4200m")
files=sorted(C4.glob("C4_200M.tsv-*")) or sorted(Path("/kaggle/input").rglob("C4_200M.tsv-*"))
assert files,"Attach the c4200m dataset"
need=TRAIN_N+VAL_N+EVAL_N; rows=[]; seen=set()
for f in files:
    kept=0
    for ch in pd.read_csv(f,sep="\t",header=None,names=["src","tgt"],dtype=str,quoting=3,
                          on_bad_lines="skip",chunksize=200_000):
        for s,t in zip(ch["src"],ch["tgt"]):
            if not isinstance(s,str) or not isinstance(t,str): continue
            s,t=norm_ws(s),norm_ws(t)
            if clean_reason(s,t) or s.lower() in seen: continue
            seen.add(s.lower()); rows.append((s,t)); kept+=1
            if kept>=PER_SHARD: break
        if kept>=PER_SHARD or len(rows)>=need: break
    if len(rows)>=need: break
random.Random(SEED).shuffle(rows); rows=rows[:need]
TR,VA,TE = rows[:TRAIN_N], rows[TRAIN_N:TRAIN_N+VAL_N], rows[TRAIN_N+VAL_N:need]
print(f"train {len(TR):,} val {len(VA):,} eval {len(TE):,}")

class PairDS(Dataset):
    def __init__(self,p,tok,dp=0.0,seed=SEED):
        self.p,self.tok,self.dp,self.rng=p,tok,dp,random.Random(seed)
    def __len__(self): return len(self.p)
    def __getitem__(self,i):
        s,t=self.p[i]
        if self.dp>0 and s!=t:
            w=[x for x in s.split() if self.rng.random()>self.dp]; s=" ".join(w) if w else s
        x=self.tok(PREFIX+s,max_length=MAX_INPUT_LEN,truncation=True)
        x["labels"]=self.tok(text_target=t,max_length=MAX_TARGET_LEN,truncation=True)["input_ids"]
        return x


In [ ]:
def tok_edit(a,b):
    a,b=a.split(),b.split(); prev=list(range(len(b)+1))
    for i,x in enumerate(a,1):
        cur=[i]
        for j,y in enumerate(b,1): cur.append(min(prev[j]+1,cur[j-1]+1,prev[j-1]+(x!=y)))
        prev=cur
    return prev[-1]
def ngrams(x,n): return Counter(tuple(x[i:i+n]) for i in range(len(x)-n+1))
def gleu(src,hyp,ref,N=4):
    num=den=0.0
    for s,h,r in zip(src,hyp,ref):
        s,h,r=s.split(),h.split(),r.split()
        for n in range(1,N+1):
            hn,rn,sn=ngrams(h,n),ngrams(r,n),ngrams(s,n)
            num+=max(sum((hn&rn).values())-(sum((hn&sn).values())-sum((hn&sn&rn).values())),0)
            den+=max(sum(hn.values()),1)
    bl=min(1.0,math.exp(1-sum(len(r.split()) for r in ref)/max(sum(len(h.split()) for h in hyp),1)))
    return bl*(num/max(den,1))
def outcome_table(srcs,tgts,preds):
    c=Counter(); n=len(srcs)
    for s,t,p in zip(srcs,tgts,preds):
        ds,dp=tok_edit(s,t),tok_edit(p,t)
        if p.strip()==t.strip(): c["perfect"]+=1
        elif p.strip()==s.strip(): c["unchanged"]+=1
        elif dp<ds: c["improved"]+=1
        elif dp==ds: c["lateral"]+=1
        else: c["harm"]+=1
    o={k:100*c[k]/n for k in ("perfect","improved","unchanged","lateral","harm")}
    o["harm_bundled_old"]=o["harm"]+o["lateral"]; o["helpful"]=o["perfect"]+o["improved"]; return o
import errant, spacy
_A={"a":None}
def ann():
    if _A["a"] is None:
        try: nlp=spacy.load("en_core_web_sm")
        except Exception:
            subprocess.run([sys.executable,"-m","spacy","download","en_core_web_sm"],check=False)
            nlp=spacy.load("en_core_web_sm")
        try: _A["a"]=errant.load("en",nlp)
        except TypeError: _A["a"]=errant.load("en")
    return _A["a"]
def errant_prf(srcs,hyps,refs):
    a=ann(); tp=fp=fn=0
    for s,h,r in zip(srcs,hyps,refs):
        so=a.parse(s)
        hyp={(e.o_start,e.o_end,e.c_str) for e in a.annotate(so,a.parse(h)) if e.type!="noop"}
        gold={(e.o_start,e.o_end,e.c_str) for e in a.annotate(so,a.parse(r)) if e.type!="noop"}
        tp+=len(hyp&gold); fp+=len(hyp-gold); fn+=len(gold-hyp)
    P=tp/max(tp+fp,1); R=tp/max(tp+fn,1)
    return dict(P=P,R=R,F05=(1.25*P*R/max(0.25*P+R,1e-9)) if (P+R) else 0.0,tp=tp,fp=fp,fn=fn)

@torch.no_grad()
def generate(model,tok,texts,bs=GEN_BATCH,desc="",gen=None):
    model.eval(); gk=dict(BASE_GEN); gk.update(gen or {}); out=[]
    for i in range(0,len(texts),bs):
        enc=tok([PREFIX+t for t in texts[i:i+bs]],return_tensors="pt",padding=True,
                truncation=True,max_length=MAX_INPUT_LEN).to(model.device)
        out+=tok.batch_decode(model.generate(**enc,**gk),skip_special_tokens=True)
        if desc and i%(bs*20)==0: print(f"    {desc} {i}/{len(texts)}",flush=True)
    return [norm_ws(x) for x in out]

def train_t5(tag,use_gate=False,seed=SEED,pairs=None,epochs=EPOCHS,lr=LEARNING_RATE,init=None,save_to=None):
    seed_all(seed); pairs = TR if pairs is None else pairs
    if init is not None: tok,model=init
    else:
        tok=AutoTokenizer.from_pretrained("google/flan-t5-small")
        model=(T5WithCopyGate if use_gate else AutoModelForSeq2SeqLM).from_pretrained("google/flan-t5-small")
    targs=dict(output_dir=str(OUT/f"ck_{tag}"),per_device_train_batch_size=PER_DEVICE_BATCH,
               gradient_accumulation_steps=GRAD_ACCUM,num_train_epochs=epochs,learning_rate=lr,
               label_smoothing_factor=LABEL_SMOOTHING,lr_scheduler_type="cosine",warmup_ratio=0.03,
               weight_decay=0.01,logging_steps=200,save_strategy="no",fp16=torch.cuda.is_available(),
               report_to=[],seed=seed,dataloader_num_workers=2,predict_with_generate=False)
    import inspect
    p=inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
    targs["eval_strategy" if "eval_strategy" in p else "evaluation_strategy"]="no"
    tr=Seq2SeqTrainer(model=model,args=Seq2SeqTrainingArguments(**targs),
                      train_dataset=PairDS(pairs,tok,SRC_DROPOUT_P,seed),
                      data_collator=DataCollatorForSeq2Seq(tok,model=model))
    t0=time.time(); o=tr.train(); dt=time.time()-t0
    assert o.training_loss>1e-6, f"{tag}: loss is 0 - model not learning!"
    if save_to: model.save_pretrained(save_to); tok.save_pretrained(save_to)
    del tr; gc.collect()
    return tok,model,dict(steps=int(o.global_step),train_loss=float(o.training_loss),
                          train_seconds=round(dt,1),
                          global_batch=PER_DEVICE_BATCH*GRAD_ACCUM*max(torch.cuda.device_count(),1))


## PHASE 1 - copy gate, FIXED (R2.6)
Pre-flight first: 30 steps in each setting. It **proves** the `label_smoothing` bug and the fix,
and asserts before spending GPU hours.

In [ ]:
if RUN_P1_GATE:
    def tiny(use_gate,ls,steps=30):
        seed_all(SEED)
        tok=AutoTokenizer.from_pretrained("google/flan-t5-small")
        m=(T5WithCopyGate if use_gate else AutoModelForSeq2SeqLM).from_pretrained("google/flan-t5-small")
        targs=dict(output_dir=str(OUT/"tmp_pf"),per_device_train_batch_size=8,max_steps=steps,
                   learning_rate=LEARNING_RATE,label_smoothing_factor=ls,logging_steps=10,
                   save_strategy="no",fp16=torch.cuda.is_available(),report_to=[],seed=SEED,
                   dataloader_num_workers=0)
        import inspect
        p=inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
        targs["eval_strategy" if "eval_strategy" in p else "evaluation_strategy"]="no"
        tr=Seq2SeqTrainer(model=m,args=Seq2SeqTrainingArguments(**targs),
                          train_dataset=PairDS(TR[:400],tok,0.0,SEED),
                          data_collator=DataCollatorForSeq2Seq(tok,model=m))
        o=tr.train(); pg=getattr(m,"_last_pgen_mean",None)
        del m,tr; gc.collect(); torch.cuda.empty_cache()
        return float(o.training_loss),pg
    print("--- PRE-FLIGHT ---")
    lb,pb=tiny(True,0.1);  print(f"  gate  + label_smoothing=0.1 -> loss={lb:.5f} p_gen={pb}")
    lf,pf=tiny(True,0.0);  print(f"  gate  + label_smoothing=0.0 -> loss={lf:.5f} p_gen={pf}")
    ln,_ =tiny(False,0.0); print(f"  nogate+ label_smoothing=0.0 -> loss={ln:.5f}")
    RESULTS["preflight"]=dict(gate_ls01=lb,gate_ls00=lf,nogate_ls00=ln,pgen_ls01=pb,pgen_ls00=pf); save()
    assert lf>1e-6,"FIX FAILED - gate loss still 0. Stop and report."
    print("PRE-FLIGHT OK\n")


In [ ]:
def run_gate(tag,use_gate,seed):
    if tag in RESULTS: print(f"== {tag}: done"); return
    if not budget_ok(0.7): return
    print(f"\n{'='*64}\n== {tag} ({hours_left():.2f}h left)\n{'='*64}",flush=True)
    tok,model,meta=train_t5(tag,use_gate=use_gate,seed=seed)
    srcs=[s for s,_ in TE]; tgts=[t for _,t in TE]
    preds=generate(model,tok,srcs,desc=tag)
    r=outcome_table(srcs,tgts,preds); r.update(errant_prf(srcs,preds,tgts)); r["gleu"]=gleu(srcs,preds,tgts)
    RESULTS[tag]=dict(tag=tag,use_gate=use_gate,seed=seed,label_smoothing=LABEL_SMOOTHING,
                      mean_p_gen=getattr(model,"_last_pgen_mean",None),**meta,**r); save()
    pd.DataFrame({"src":srcs,"tgt":tgts,"pred":preds}).to_csv(OUT/f"revBC_preds_{tag}.csv",index=False)
    print(f"-> loss={meta['train_loss']:.4f} F0.5={r['F05']:.4f} P={r['P']:.3f} R={r['R']:.3f} "
          f"harm={r['harm']:.2f}% p_gen={RESULTS[tag]['mean_p_gen']}")
    del model; gc.collect(); torch.cuda.empty_cache()

if RUN_P1_GATE:
    for s in SEEDS:
        run_gate(f"nogate_s{s}",False,s)
        run_gate(f"gate_s{s}",  True, s)


## PHASE 2 - stage-2 vs joint, judged on **learner text** (R2.7)

In [ ]:
def load_wi():
    d=OUT/"bea19"; d.mkdir(exist_ok=True)
    if not (d/"wi+locness").exists():
        urllib.request.urlretrieve(BEA_URL,d/"wi.tar.gz")
        with tarfile.open(d/"wi.tar.gz") as t: t.extractall(d)
    return d
def m2_to_pairs(path, ann_id=0):
    """returns (src, tgt, cefr_tag) using annotator `ann_id`."""
    out=[]; src=None; edits=[]
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if line.startswith("S "):
            if src is not None: out.append((src,edits)); edits=[]
            src=line[2:].strip()
        elif line.startswith("A "):
            p=line[2:].split("|||")
            if p[1] in ("noop","UNK"): continue
            if int(p[-1])!=ann_id: continue
            a,b=p[0].split(); edits.append((int(a),int(b),p[2]))
    if src is not None: out.append((src,edits))
    pairs=[]
    for s,e in out:
        w=s.split()
        for a,b,rep in sorted(e,key=lambda x:-x[0]): w[a:b]=rep.split() if rep else []
        pairs.append((norm_ws(s),norm_ws(" ".join(w))))
    return pairs

if RUN_P2_TRANSFER and budget_ok(1.2):
    d=load_wi(); m2d=d/"wi+locness"/"m2"
    WI=[]
    for f in sorted(m2d.glob("*.train.gold.bea19.m2")): WI+=m2_to_pairs(f)
    random.Random(SEED).shuffle(WI)
    WI_HOLD,WI_TRAIN=WI[:int(len(WI)*0.05)],WI[int(len(WI)*0.05):]
    BEA_DEV=m2_to_pairs(m2d/"ABCN.dev.gold.bea19.m2")
    print(f"W&I train {len(WI_TRAIN):,} | holdout {len(WI_HOLD):,} | BEA dev {len(BEA_DEV):,}")
    if QUICK_TEST: WI_TRAIN=WI_TRAIN[:2000]; BEA_DEV=BEA_DEV[:300]
    BEA_S=[s for s,_ in BEA_DEV]; BEA_T=[t for _,t in BEA_DEV]

    BASE_DIR=OUT/"base_model"
    if not BASE_DIR.exists():
        tok,model,meta=train_t5("base_transfer",seed=SEED,save_to=BASE_DIR)
        del model; gc.collect(); torch.cuda.empty_cache()

    def eval_both(tag,tok,model):
        srcs=[s for s,_ in TE]; tgts=[t for _,t in TE]
        pin=generate(model,tok,srcs,desc=f"{tag}-indomain")
        rin=outcome_table(srcs,tgts,pin); rin.update(errant_prf(srcs,pin,tgts))
        pb=generate(model,tok,BEA_S,desc=f"{tag}-BEA")
        rb=outcome_table(BEA_S,BEA_T,pb); rb.update(errant_prf(BEA_S,pb,BEA_T))
        RESULTS[tag]={"tag":tag,**{f"indomain_{k}":v for k,v in rin.items()},
                                  **{f"bea_{k}":v for k,v in rb.items()}}; save()
        pd.DataFrame({"src":BEA_S,"tgt":BEA_T,"pred":pb}).to_csv(OUT/f"revBC_preds_{tag}_bea.csv",index=False)
        print(f"-> {tag}: in-domain F0.5={rin['F05']:.4f} | BEA F0.5={rb['F05']:.4f} "
              f"(P={rb['P']:.3f} R={rb['R']:.3f})")

    if "transfer_base" not in RESULTS:
        tok=AutoTokenizer.from_pretrained(str(BASE_DIR)); m=AutoModelForSeq2SeqLM.from_pretrained(str(BASE_DIR)).to(DEV)
        eval_both("transfer_base",tok,m); del m; gc.collect(); torch.cuda.empty_cache()
    if "transfer_stage2" not in RESULTS and budget_ok(0.8):
        tok=AutoTokenizer.from_pretrained(str(BASE_DIR)); m=AutoModelForSeq2SeqLM.from_pretrained(str(BASE_DIR))
        tok,m,_=train_t5("stage2",init=(tok,m),pairs=WI_TRAIN,epochs=STAGE2_EPOCHS,lr=STAGE2_LR,
                         save_to=OUT/"stage2_model")
        eval_both("transfer_stage2",tok,m); del m; gc.collect(); torch.cuda.empty_cache()
    if "transfer_joint" not in RESULTS and budget_ok(0.8):
        k=min(len(WI_TRAIN),TRAIN_N//4)
        mixed=TR[:TRAIN_N-k]+WI_TRAIN[:k]; random.Random(SEED).shuffle(mixed)
        tok,m,_=train_t5("joint",pairs=mixed,seed=SEED)
        eval_both("transfer_joint",tok,m); del m; gc.collect(); torch.cuda.empty_cache()


## PHASE 3 - **official CoNLL-2014 M² scorer** (R2.3)
The reviewer called mixing ERRANT and M² numbers *"potentially fatal"*. Here we compute **both**
scorers on the **same** system output, so the paper can report them side by side.

In [ ]:
def get_conll():
    d=OUT/"conll14"; d.mkdir(exist_ok=True)
    m2=d/"conll14st-test-data"/"noalt"/"official-2014.combined.m2"
    if not m2.exists():
        urllib.request.urlretrieve(CONLL_URL,d/"c.tar.gz")
        with tarfile.open(d/"c.tar.gz") as t: t.extractall(d)
    srcs=[l[2:].strip() for l in m2.read_text(encoding="utf-8").splitlines() if l.startswith("S ")]
    return m2,srcs

def setup_m2scorer():
    """official m2scorer is python2 -> download + 2to3. Returns path to the script or None."""
    d=OUT/"m2s"
    if (d/"m2scorer"/"scripts"/"m2scorer.py").exists(): return d/"m2scorer"/"scripts"/"m2scorer.py"
    d.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(M2_URL,d/"m2scorer.tar.gz")
        with tarfile.open(d/"m2scorer.tar.gz") as t: t.extractall(d)
        r=subprocess.run([sys.executable,"-m","lib2to3","-w","-n","--no-diffs",str(d)],
                         capture_output=True,text=True)
        print("  2to3 rc:",r.returncode)
        cand=list(d.rglob("m2scorer.py"))
        cand=[c for c in cand if "scripts" in str(c)] or cand
        return cand[0] if cand else None
    except Exception as e:
        print("  m2scorer setup failed:",e); return None

def official_m2(hyps, gold_m2, tag):
    n_gold=sum(1 for l in Path(gold_m2).read_text(encoding="utf-8").splitlines() if l.startswith("S "))
    if len(hyps)!=n_gold:
        print(f"  skip official M2 for {tag}: {len(hyps)} hyps vs {n_gold} gold sentences")
        return None
    s=setup_m2scorer()
    if s is None: return None
    hp=OUT/f"{tag}.hyp.txt"; hp.write_text("\n".join(hyps)+"\n",encoding="utf-8")
    try:
        r=subprocess.run([sys.executable,str(s),str(hp),str(gold_m2)],capture_output=True,text=True,timeout=1800)
        txt=r.stdout+r.stderr
        got={}
        for line in txt.splitlines():
            m=re.match(r"\s*(Precision|Recall|F_?0\.5)\s*:?\s*([0-9.]+)",line,re.I)
            if m: got[m.group(1).lower().replace("_","")]=float(m.group(2))
        if got: return dict(P=got.get("precision"),R=got.get("recall"),F05=got.get("f0.5"),raw=txt[-400:])
        print("  m2scorer output not parsed:",txt[-400:]); return None
    except Exception as e:
        print("  m2scorer run failed:",e); return None

if RUN_P3_M2 and budget_ok(0.5):
    C_M2,C_SRC=get_conll()
    if QUICK_TEST: C_SRC=C_SRC[:100]
    print(f"CoNLL-2014 test: {len(C_SRC)} sentences")
    # (a) our matched-budget model
    BASE_DIR=OUT/"base_model"
    if BASE_DIR.exists() and "conll_ours_matched" not in RESULTS:
        tok=AutoTokenizer.from_pretrained(str(BASE_DIR)); m=AutoModelForSeq2SeqLM.from_pretrained(str(BASE_DIR)).to(DEV)
        cp=generate(m,tok,C_SRC,desc="conll-ours")
        pd.DataFrame({"src":C_SRC,"pred":cp}).to_csv(OUT/"revBC_conll_ours_matched.csv",index=False)
        off=official_m2(cp,C_M2,"ours_matched")
        RESULTS["conll_ours_matched"]=dict(tag="conll_ours_matched",official_M2=off); save()
        print("  official M2 (ours, matched budget):",off if off else "unavailable")
        del m; gc.collect(); torch.cuda.empty_cache()
    # (b) FULL-SCALE paper system: predictions are EMBEDDED above (no dataset needed)
    if "conll_ours_fullscale" not in RESULTS:
        preds=[norm_ws(x) for x in FULL_CONLL_PRED]
        _M2_SRC=[l[2:].strip() for l in Path(C_M2).read_text(encoding="utf-8").splitlines() if l.startswith("S ")]
        print(f"  full-scale paper predictions: {len(preds)} (gold {len(_M2_SRC)})")
        off=official_m2(preds,C_M2,"ours_full")
        err=errant_prf(_M2_SRC,preds,_M2_SRC) if False else None   # ERRANT needs refs, not src
        RESULTS["conll_ours_fullscale"]=dict(tag="conll_ours_fullscale",n=len(preds),official_M2=off,
            note=("THE number for the paper: identical outputs to the reported ERRANT F0.5=0.453, "
                  "now also scored with the official CoNLL-2014 M2 scorer (reviewer R2.3)")); save()
        print("  >>> official M2 (FULL-SCALE paper system):",off if off else "unavailable")
        # do-nothing baseline under the same scorer, for context
        if "conll_donothing" not in RESULTS:
            off0=official_m2(_M2_SRC,C_M2,"donothing")
            RESULTS["conll_donothing"]=dict(tag="conll_donothing",official_M2=off0); save()
            print("  do-nothing baseline (official M2):",off0)


## PHASE 4 - 2024 LLM baseline, identical splits + identical scorers (R1.4 / R1.10 / R2.1)

In [ ]:
FEWSHOT=[("She go to the store yesterday and buy some milk.",
          "She went to the store yesterday and bought some milk."),
         ("I am interested at environment and ecological things.",
          "I am interested in the environment and ecological things."),
         ("What other precautions , is usually taken ?",
          "What other precautions are usually taken ?"),
         ("There are many local foods very popular .",
          "There are many very popular local foods .")]
def llm_prompts(tok,srcs,shots):
    out=[]
    for s in srcs:
        msgs=[{"role":"system","content":"You are a grammatical error correction system. "
               "Return ONLY the corrected sentence. Make the minimum number of edits. "
               "If the sentence is already correct, repeat it unchanged."}]
        for a,b in shots:
            msgs.append({"role":"user","content":a}); msgs.append({"role":"assistant","content":b})
        msgs.append({"role":"user","content":s})
        out.append(tok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True))
    return out

@torch.no_grad()
def llm_generate(model,tok,srcs,shots,desc=""):
    prompts=llm_prompts(tok,srcs,shots); outs=[]
    for i in range(0,len(prompts),LLM_BATCH):
        b=prompts[i:i+LLM_BATCH]
        enc=tok(b,return_tensors="pt",padding=True).to(model.device)
        g=model.generate(**enc,max_new_tokens=LLM_MAX_NEW,do_sample=False,
                         pad_token_id=tok.pad_token_id or tok.eos_token_id)
        for j in range(len(b)):
            txt=tok.decode(g[j][enc.input_ids.shape[1]:],skip_special_tokens=True)
            outs.append(norm_ws(txt.split("\n")[0]))
        if i%(LLM_BATCH*10)==0: print(f"    {desc} {i}/{len(prompts)}",flush=True)
    return outs

if RUN_P4_LLM and budget_ok(1.5):
    print(f"loading {LLM_NAME} ...")
    ltok=AutoTokenizer.from_pretrained(LLM_NAME)
    if ltok.pad_token is None: ltok.pad_token=ltok.eos_token
    ltok.padding_side="left"   # decoder-only models need left padding
    lm=AutoModelForCausalLM.from_pretrained(LLM_NAME,torch_dtype=torch.float16,device_map="auto")
    lm.eval()
    C_M2,C_SRC=get_conll()
    if QUICK_TEST: C_SRC=C_SRC[:60]
    for name,shots in [("zeroshot",[]),("fewshot",FEWSHOT)]:
        tag=f"llm_conll_{name}"
        if tag in RESULTS or not budget_ok(0.5): continue
        p=llm_generate(lm,ltok,C_SRC,shots,desc=tag)
        pd.DataFrame({"src":C_SRC,"pred":p}).to_csv(OUT/f"revBC_{tag}.csv",index=False)
        off=official_m2(p,C_M2,tag)
        RESULTS[tag]=dict(tag=tag,model=LLM_NAME,shots=len(shots),official_M2=off); save()
        print(f"-> {tag}: official M2 = {off}")
    # BEA dev (ERRANT, same protocol as ours)
    try:
        d=load_wi(); BEA=m2_to_pairs(d/"wi+locness"/"m2"/"ABCN.dev.gold.bea19.m2")
        if QUICK_TEST: BEA=BEA[:60]
        bs=[s for s,_ in BEA]; bt=[t for _,t in BEA]
        for name,shots in [("fewshot",FEWSHOT)]:
            tag=f"llm_bea_{name}"
            if tag in RESULTS or not budget_ok(0.6): continue
            p=llm_generate(lm,ltok,bs,shots,desc=tag)
            r=errant_prf(bs,p,bt); r.update(outcome_table(bs,bt,p))
            pd.DataFrame({"src":bs,"tgt":bt,"pred":p}).to_csv(OUT/f"revBC_{tag}.csv",index=False)
            RESULTS[tag]=dict(tag=tag,model=LLM_NAME,shots=len(shots),**r); save()
            print(f"-> {tag}: ERRANT F0.5={r['F05']:.4f} P={r['P']:.3f} R={r['R']:.3f} harm={r['harm']:.2f}%")
    except Exception as e: print("BEA LLM eval skipped:",e)
    del lm; gc.collect(); torch.cuda.empty_cache()


## PHASE 5 - BEA-2019 dev per CEFR level (R2.7)

In [ ]:
if RUN_P5_CEFR and budget_ok(0.4):
    d=load_wi(); m2d=d/"wi+locness"/"m2"
    lv={}
    for f in sorted(m2d.glob("*.dev.gold.bea19.m2")):
        nm=f.name.split(".")[0]
        if nm in ("A","B","C","N"): lv[nm]=m2_to_pairs(f)
    print("levels found:",{k:len(v) for k,v in lv.items()})
    # (0) FULL-SCALE paper system, using the embedded BEA dev predictions
    if "cefr_fullscale_paper" not in RESULTS:
        ABCN=m2_to_pairs(m2d/"ABCN.dev.gold.bea19.m2")
        if len(ABCN)==len(FULL_BEA_PRED):
            src2pred={s:p for (s,_),p in zip(ABCN,[norm_ws(x) for x in FULL_BEA_PRED])}
            per={}
            for k,pairs in lv.items():
                s=[x for x,_ in pairs]; t=[y for _,y in pairs]
                pr=[src2pred.get(x,x) for x in s]
                cov=sum(1 for x in s if x in src2pred)/max(len(s),1)
                r=errant_prf(s,pr,t); r.update(outcome_table(s,t,pr))
                per[k]=dict(n=len(pairs),coverage=round(cov,3),**r)
                print(f"   FULL-SCALE level {k}: F0.5={r['F05']:.4f} n={len(pairs)} cov={cov:.2f}")
            RESULTS["cefr_fullscale_paper"]=dict(tag="cefr_fullscale_paper",per_level=per,
                note="per-CEFR breakdown of the system reported in the paper (BEA dev ERRANT F0.5=0.429)")
            save()
        else:
            print(f"   !! ABCN dev {len(ABCN)} != embedded {len(FULL_BEA_PRED)} - skipping full-scale CEFR")
    BASE_DIR=OUT/"base_model"; S2=OUT/"stage2_model"
    for mdir,mtag in [(S2,"stage2"),(BASE_DIR,"base")]:
        if not Path(mdir).exists(): continue
        tag=f"cefr_{mtag}"
        if tag in RESULTS or not budget_ok(0.3): continue
        tok=AutoTokenizer.from_pretrained(str(mdir)); m=AutoModelForSeq2SeqLM.from_pretrained(str(mdir)).to(DEV)
        per={}
        for k,pairs in lv.items():
            if QUICK_TEST: pairs=pairs[:80]
            s=[x for x,_ in pairs]; t=[y for _,y in pairs]
            pr=generate(m,tok,s,desc=f"{tag}-{k}")
            r=errant_prf(s,pr,t); r.update(outcome_table(s,t,pr))
            per[k]=dict(n=len(pairs),**r); print(f"   {mtag} level {k}: F0.5={r['F05']:.4f} n={len(pairs)}")
        RESULTS[tag]=dict(tag=tag,per_level=per); save()
        del m; gc.collect(); torch.cuda.empty_cache()


## Bootstrap + export

In [ ]:
def paired_bootstrap(fa,fb,n_boot=1000,seed=SEED):
    fa,fb=Path(fa),Path(fb)
    if not (fa.exists() and fb.exists()): return None
    A,B=pd.read_csv(fa),pd.read_csv(fb)
    srcs=A["src"].astype(str).tolist(); tgts=A["tgt"].astype(str).tolist()
    pa=A["pred"].astype(str).tolist(); pb=B["pred"].astype(str).tolist()
    a=ann(); per=[]
    for s,t,x,y in zip(srcs,tgts,pa,pb):
        so=a.parse(s)
        gold={(e.o_start,e.o_end,e.c_str) for e in a.annotate(so,a.parse(t)) if e.type!="noop"}
        row=[]
        for p in (x,y):
            hyp={(e.o_start,e.o_end,e.c_str) for e in a.annotate(so,a.parse(p)) if e.type!="noop"}
            row.append((len(hyp&gold),len(hyp-gold),len(gold-hyp)))
        per.append(row)
    def f05(c):
        tp,fp,fn=c; P=tp/max(tp+fp,1); R=tp/max(tp+fn,1)
        return (1.25*P*R/max(0.25*P+R,1e-9)) if (P+R) else 0.0
    tot=lambda i,ids: tuple(sum(per[k][i][j] for k in ids) for j in range(3))
    obs=f05(tot(0,range(len(per))))-f05(tot(1,range(len(per))))
    rng=np.random.default_rng(seed); dd=[]
    for _ in range(n_boot):
        ids=rng.integers(0,len(per),len(per)); dd.append(f05(tot(0,ids))-f05(tot(1,ids)))
    dd=np.array(dd)
    return dict(a=fa.stem,b=fb.stem,delta_F05=obs,ci_low=float(np.percentile(dd,2.5)),
                ci_high=float(np.percentile(dd,97.5)),
                p_value=float(min(1.0,2*min((dd<=0).mean(),(dd>=0).mean()))))
BOOT=[]
for s in SEEDS:
    r=paired_bootstrap(OUT/f"revBC_preds_nogate_s{s}.csv",OUT/f"revBC_preds_gate_s{s}.csv")
    if r: BOOT.append(r); print(f"gate seed {s}: dF0.5={r['delta_F05']:+.4f} "
                                f"95%CI[{r['ci_low']:+.4f},{r['ci_high']:+.4f}] p={r['p_value']:.3f}")
r=paired_bootstrap(OUT/"revBC_preds_transfer_stage2_bea.csv",OUT/"revBC_preds_transfer_joint_bea.csv")
if r: BOOT.append(r); print(f"stage2 vs joint on BEA: dF0.5={r['delta_F05']:+.4f} p={r['p_value']:.3f}")
json.dump(BOOT,open(OUT/"revBC_bootstrap.json","w"),indent=1)

g=[RESULTS[f"gate_s{s}"]["F05"] for s in SEEDS if f"gate_s{s}" in RESULTS]
n=[RESULTS[f"nogate_s{s}"]["F05"] for s in SEEDS if f"nogate_s{s}" in RESULTS]
if g and n:
    summ=dict(nogate_mean=float(np.mean(n)),nogate_std=float(np.std(n,ddof=1)) if len(n)>1 else 0.,
              gate_mean=float(np.mean(g)),gate_std=float(np.std(g,ddof=1)) if len(g)>1 else 0.,
              delta=float(np.mean(n)-np.mean(g)),n_seeds=len(g),
              gate_pgen=[RESULTS[f"gate_s{s}"].get("mean_p_gen") for s in SEEDS if f"gate_s{s}" in RESULTS])
    json.dump(summ,open(OUT/"revBC_gate_summary.json","w"),indent=1); print("\nGATE:",summ)
pd.DataFrame(RESULTS).T.to_csv(OUT/"revBC_all.csv")
with zipfile.ZipFile(OUT/"revBC_results.zip","w",zipfile.ZIP_DEFLATED) as z:
    for pat in ["revBC_*.json","revBC_*.csv"]:
        for p in OUT.glob(pat):
            if p.suffix!=".zip": z.write(p,p.name)
print("\nDONE -> revBC_results.zip |",f"{hours_left():.2f}h spare")
